In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# Change to project root directory (parent of scripts folder)
current_dir = Path.cwd()
if current_dir.name == 'scripts':
    os.chdir(current_dir.parent)


In [ ]:
from lib.agentic.graph import AgenticGraph
from lib.agentic.streaming import agentic_rag_stream
from lib.agentic.config import get_agent_state_default

app = AgenticGraph().build_workflow().compile()


In [ ]:
# Test streaming agentic_rag - RAG path
state1 = get_agent_state_default(
    max_iter=2, 
    chunk_k=12, 
    max_query_expand_k=2
)
state1["question"] = "中国在哪些地方违反了国际法？"

print("Starting streaming agentic_rag...")
print("=" * 60)

full_content = ""
async for chunk in agentic_rag_stream(app, state1):
    chunk_type = chunk.get("type")
    chunk_data = chunk.get("data")
    
    if chunk_type == "status":
        status = chunk_data.get("status", "")
        node = chunk_data.get("node", "")
        iteration = chunk_data.get("iteration", "")
        if iteration:
            print(f"\n[{status.upper()}] Node: {node}, Iteration: {iteration}")
        else:
            print(f"\n[{status.upper()}] Node: {node}")
    
    elif chunk_type == "content":
        # Stream content chunks
        content_chunk = chunk_data
        full_content += content_chunk
        print(content_chunk, end="", flush=True)
    
    elif chunk_type == "done":
        print("\n" + "=" * 60)
        print("\n[COMPLETE] Final results:")
        print(f"Content length: {len(chunk_data.get('content', ''))}")
        print(f"Search results count: {len(chunk_data.get('search_results', []))}")
        print(f"Query type: {chunk_data.get('query_type', '')}")
        print(f"Search count: {chunk_data.get('search_count', 0)}")
        print(f"Historical search ops: {len(chunk_data.get('historical_search_ops', []))}")
    
    elif chunk_type == "error":
        print(f"\n[ERROR] {chunk_data.get('error', 'Unknown error')}")

print("\n" + "=" * 60)
print("Streaming test completed!")


In [ ]:
# Test streaming with greeting (direct answer, no RAG)
state2 = get_agent_state_default()
state2["question"] = "你好呀"

print("Testing streaming with greeting query...")
print("=" * 60)

async for chunk in agentic_rag_stream(app, state2):
    chunk_type = chunk.get("type")
    chunk_data = chunk.get("data")
    
    if chunk_type == "status":
        status = chunk_data.get("status", "")
        node = chunk_data.get("node", "")
        print(f"\n[{status.upper()}] Node: {node}")
    
    elif chunk_type == "content":
        print(chunk_data, end="", flush=True)
    
    elif chunk_type == "done":
        print("\n" + "=" * 60)
        print("\n[COMPLETE]")
        print(f"Query type: {chunk_data.get('query_type', '')}")
        print(f"Content: {chunk_data.get('content', '')[:100]}...")

print("\n" + "=" * 60)


In [ ]:
# Compare streaming vs non-streaming results
import asyncio

state3 = get_agent_state_default(max_iter=2, chunk_k=12, max_query_expand_k=2)
state3["question"] = "什么是人工智能？"

print("Comparing streaming vs non-streaming...")
print("=" * 60)

# Non-streaming (original)
print("\n[Non-streaming]")
result_non_stream = await app.ainvoke(state3.copy())
print(f"Answer length: {len(result_non_stream.get('answer', ''))}")
print(f"Answer preview: {result_non_stream.get('answer', '')[:200]}...")

# Streaming
print("\n[Streaming]")
state3_stream = state3.copy()
streamed_content = ""
async for chunk in agentic_rag_stream(app, state3_stream):
    if chunk.get("type") == "content":
        streamed_content += chunk.get("data", "")
    elif chunk.get("type") == "done":
        streamed_answer = chunk.get("data", {}).get("content", "")
        print(f"Answer length: {len(streamed_answer)}")
        print(f"Answer preview: {streamed_answer[:200]}...")
        break

print("\n" + "=" * 60)
print("Comparison completed!")
